# Kaggle diabetes prediction challenge: Submission template

Our lesson 20: classification activity will use an active Kaggle playground competition. Playground competitions run every month and highlight interesting/approachable datasets for the ML community to practice skills on a variety of ML problem types without the pressure, complexity and long duration of prize money competitions. 

It just so happens that this month's playground competition is a classification challenge! See the full competition details here: [Playground Series - Season 5, Episode 12: Diabetes Prediction Challenge](https://www.kaggle.com/competitions/playground-series-s5e12)

This notebook template will help you make your first Kaggle competition submission. It is pre-filled with code to load the competition data and output random predictions (see section 5. Submission below). This will score ~0.50 on the Kaggle public leaderboard. Your job is to improve that score with EDA, clever feature engineering and good model optimization. Good luck!

## How to submit on Kaggle

**1. Upload this notebook to Kaggle:**
   - Create a Kaggle account and log in to [kaggle](https://www.kaggle.com)
   - Click '+ Create' in the left navigation menu
   - Select 'Import Notebook' and upload this file (or link to GitHub)

**2. Start the notebook:**
   - Find your uploaded notebook in your Kaggle account under 'Code'
   - Click on the notebook to open it
   - The notebook will open in edit mode - you can now run cells and make changes

**3. Add the competition dataset:**
   - From the notebook environment, in the right sidebar, click the 'Input' tab
   - Click '+ Add Input' → filter by 'Competition Datasets'
   - Find 'Diabetes Prediction Challenge' and click the '+' icon
   - Note: You must join the competition first (click 'Join Competition' on the competition page)

**4. Access the data:**
   - Once added, the data is available at `/kaggle/input/playground-series-s5e12/`
   - Files: `train.csv` (training data with labels), `test.csv` (test data without labels), `sample_submission.csv` (submission format example)

**5. Make your submission:**
   - Your notebook must output test set predictions to `submission.csv` in the correct format
   - Go to 'Submit to competition' tab in the right sidebar and click 'Submit'

**Note:** This notebook uses a `KAGGLE` flag (under 'Run configuration') to switch between Kaggle and local file paths. Set it to `True` when running on Kaggle, or `False` when running locally with data in a `../data/` directory.

## Notebook set-up

### Imports

In [11]:
# Standard library imports
from pathlib import Path


import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t
from scipy.stats import boxcox, yeojohnson, shapiro
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

from sklearn.preprocessing import (
    KBinsDiscretizer,
    PowerTransformer,
    QuantileTransformer,
    MinMaxScaler,
    StandardScaler,
    OrdinalEncoder,
    OneHotEncoder,
    LabelEncoder,
    PolynomialFeatures
)
import platform
import sys
import platform
import sys

# How to tell python version
#print (sys.version_info)
#print (platform.python_version())

# How to pip install from Terminal window:
# python -m pip install seaborn

# Put this at the top of your notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)  # Set a large number
pd.set_option('display.max_colwidth', None)

# Set random seed for reproducibility
np.random.seed(315)

### Run configuration

In [12]:
# Set to True when running on Kaggle, False when running locally
KAGGLE = False

#####  **Utility Function Definitions** #####

In [13]:
def PrintDataFrameStatus(df_target):
    # Print Stats
    print("***********************************")
    print("Description Stats")
    print("***********************************")
    print()
    print(df_target.describe(include='all').T)
    print()

    # Print df Column Info
    print("***********************************")
    print("Basic Info of imported data set")
    print("***********************************")
    print()
    df_target.info()
    print()
    print()

    print(f'df_floridabikerentals Shape:{df_target.shape}')
    print()

    print('Do we have any features with null values?:')
    print(df_target.isnull().any().any())
    print()

    print('Feature Columns with that have null values:')
    print(df_target.isnull().sum()[df_target.isnull().sum() > 0])
    print()

    print('Do we have any features with nan values?:')
    print(df_target.isna().any().any())

    print("***********************************")
    print("First 20 rows of Data")
    print("***********************************")
    print()
    print(df_target.head(20))
    print()

    print("***********************************")
    print("First 20 rows of Random Sample Data")
    print("***********************************")
    print()
    print(df_target.sample(20))
    print

### Data loading

In [14]:
# Set file paths based on environment
if KAGGLE:
    # Kaggle paths - data is in /kaggle/input/
    train_df_path = '/kaggle/input/playground-series-s5e12/train.csv'
    test_df_path = '/kaggle/input/playground-series-s5e12/test.csv'

else:
    # Otherwise, load data from course GitHub repository
    train_df_path = 'diabetes_prediction_train.csv'
    test_df_path = 'diabetes_prediction_test.csv'

# Load the training and testing datasets
df_train = pd.read_csv(train_df_path)
df_test = pd.read_csv(test_df_path)

# Display first few rows of training data
#train_df.head()

##### Examine **Training Dataset** #####

In [15]:
PrintDataFrameStatus(df_train)

***********************************
Description Stats
***********************************

                                       count unique         top    freq        mean            std    min        25%       50%        75%       max
id                                  700000.0    NaN         NaN     NaN    349999.5  202072.738554    0.0  174999.75  349999.5  524999.25  699999.0
age                                 700000.0    NaN         NaN     NaN   50.359734       11.65552   19.0       42.0      50.0       58.0      89.0
alcohol_consumption_per_week        700000.0    NaN         NaN     NaN    2.072411       1.048189    1.0        1.0       2.0        3.0       9.0
physical_activity_minutes_per_week  700000.0    NaN         NaN     NaN   80.230803      51.195071    1.0       49.0      71.0       96.0     747.0
diet_score                          700000.0    NaN         NaN     NaN    5.963695       1.463336    0.1        5.0       6.0        7.0       9.9
sleep_hours_per_day  

##### Examine **Test Dataset** #####

In [16]:
PrintDataFrameStatus(df_test)

***********************************
Description Stats
***********************************

                                       count unique         top    freq        mean           std       min        25%       50%        75%       max
id                                  300000.0    NaN         NaN     NaN    849999.5  86602.684716  700000.0  774999.75  849999.5  924999.25  999999.0
age                                 300000.0    NaN         NaN     NaN   50.432397     11.938741      19.0       42.0      50.0       59.0      89.0
alcohol_consumption_per_week        300000.0    NaN         NaN     NaN    2.089693      1.066214       1.0        1.0       2.0        3.0       9.0
physical_activity_minutes_per_week  300000.0    NaN         NaN     NaN   92.349087     62.187399       1.0       51.0      77.0      115.0     748.0
diet_score                          300000.0    NaN         NaN     NaN    5.945838      1.481068       0.1        5.0       6.0        7.0       9.9
sleep_hou

## 1. EDA

#### **Potential Data Issues from Imported Data** ####
    The following columns (features) have missing data:
        There are no features with missing/null/<nan> data.
    
##### **Validate Data Features** #####
    Test each categorical feature to ensure it's context and validity

In [17]:
# Your code here...

## 2. Feature engineering

In [18]:
# Your code here...

## 3. Model building

In [19]:
# Your code here...

## 4. Model evaluation

In [20]:
# Your code here...

## 5. Submission

In [21]:
# Make random predictions for submission by sampling from the training labels
# Note: Replace this with your model's predictions!
predictions = train_df['diagnosed_diabetes'].sample(n=test_df.shape[0], random_state=42).astype(int)
prediction_ids = test_df['id'].astype(int)

# Create submission DataFrame with required format: id, diagnosed_diabetes
submission_df = pd.DataFrame({
    'id': prediction_ids.values,
    'diagnosed_diabetes': predictions.values
})

# Determine output path based on environment
if KAGGLE:

    # On Kaggle, save to current directory for submission
    submission_file = 'submission.csv'

else:

    # Locally, save to ../data/ directory
    # Create directory if it doesn't exist
    data_dir = Path('../data')
    data_dir.mkdir(parents=True, exist_ok=True)
    submission_file = data_dir / 'submission.csv'

# Save submission file and display preview
submission_df.to_csv(submission_file, index=False)
submission_df.head()

,id,diagnosed_diabetes
0,700000,1
1,700001,0
2,700002,1
3,700003,1
4,700004,0
